In [2]:
import os 
import glob # for automatic file discovery
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler # for normalization

## Loading CSVs

In [3]:
def load_all_csv(csv_folder):
    files = glob.glob(os.path.join(csv_folder,"*.csv"))
    print("Found CSV files:", files)
    df_list=[pd.read_csv(f) for f in files] # reading all the files in the csv folder
    data=pd.concat(df_list, ignore_index=True) 
    data=data.sort_values(by="frame").reset_index(drop=True)
    # setting the indices wrt frame
    return data

In [4]:
df = load_all_csv("./csv_folder")
df.head()

Found CSV files: ['./csv_folder\\session_20251207_233723.csv', './csv_folder\\session_20251207_234144.csv']


,timestamp,frame,px,py,pvx,pvy,a_attack,a_dash,a_idle,aim
0,0.020,0,0.0,0.0000,0.0,0.0,0,0,1,0.8961
1,0.020,0,0.0,0.0000,0.0,0.0,0,0,1,1.9684
2,0.068,1,0.0,-0.0827,0.0,-2.4,0,0,0,1.8711
3,0.068,1,0.0,0.0000,0.0,0.0,0,0,1,0.7988
4,0.101,2,0.0,0.0000,0.0,0.0,0,0,1,0.7915


In [5]:
df.tail()

,timestamp,frame,px,py,pvx,pvy,a_attack,a_dash,a_idle,aim
1565,27.200,815,-2.1761,-1.7432,0.0,0.0,0,0,1,1.6628
1566,27.233,816,-2.1761,-1.7432,0.0,0.0,0,0,1,1.6565
1567,27.266,817,-2.1761,-1.7432,0.0,0.0,0,0,1,1.6509
1568,27.301,818,-2.1761,-1.7432,0.0,0.0,0,0,1,1.6461
1569,27.335,819,-2.1761,-1.7432,0.0,0.0,0,0,1,1.6421


## Normalization

In [6]:
def normalize_data_fixed_ranges(df):
    feature_ranges = {
        "px": (-5.7, 5.7),
        "py": (-10.76, 10.76),
        "pvx": (-3.84, 3.84),
        "pvy": (-3.84, 3.84),
        "aim": (0.0, 360.0)
    }
    for feature, (fmin, fmax) in feature_ranges.items():
        df[feature] = (df[feature] - fmin) / (fmax - fmin)
    return df


In [7]:
df = normalize_data_fixed_ranges(df)
df.head()

,timestamp,frame,px,py,pvx,pvy,a_attack,a_dash,a_idle,aim
0,0.020,0,0.5,0.500000,0.5,0.5000,0,0,1,0.002489
1,0.020,0,0.5,0.500000,0.5,0.5000,0,0,1,0.005468
2,0.068,1,0.5,0.496157,0.5,0.1875,0,0,0,0.005197
3,0.068,1,0.5,0.500000,0.5,0.5000,0,0,1,0.002219
4,0.101,2,0.5,0.500000,0.5,0.5000,0,0,1,0.002199


## Building LSTM Data

In [8]:
len(df)

1570

In [9]:
def build_lstm_sequences(df, seq=20): #last 20 frames as i/p to LSTM
    X,Y=[],[]
    # X=all i/p sequences(20 frames each)
    # Y=all future posns
    features=["px","py","pvx","pvy","aim"] #LSTM i/p
    for i in range(len(df)-seq):
        X.append(df[features].iloc[i:i+seq].values)
        Y.append(df[["px","py"]].iloc[i+seq].values)

    return np.array(X),np.array(Y) 
    # converting list to numpy arrays

In [10]:
X_lstm,Y_lstm=build_lstm_sequences(df, 20)
X_lstm.shape,Y_lstm.shape

((1550, 20, 5), (1550, 2))

## Building Classifier Data
This becomes your supervised action classification dataset.

In [11]:
def build_classifier_dataset(df):
    X=df[["px", "py", "pvx", "pvy", "aim"]].values
    # X = array of shape (num_frames, 5) --> i/p to the classifier
    Y=[]
    for _, row in df.iterrows(): # we ignore the index using _
        if row["a_attack"] == 1:
            Y.append(1)
        elif row["a_dash"] == 1:
            Y.append(2)
        elif row["a_idle"] == 1:
            Y.append(0)
        else:
            Y.append(3) # default-->move
    return np.array(X), np.array(Y) 
    # converting list to numpy array

# Example
# frame 0 → 3 (moving)
# frame 1 → 3 (moving)
# frame 2 → 1 (attack)
# frame 3 → 0 (idle)
# frame 4 → 2 (dash)


In [12]:
X_cls, Y_cls = build_classifier_dataset(df)
X_cls.shape, Y_cls.shape


((1570, 5), (1570,))

### LSTM learns:
- “Given the last 20 frames, where will the player go next?”

### Classifier learns:
- “Given the player's current state, what action will the player take?”

## Saving
why .npy file?
- Loads instantly
- Preserves shape
- Preserves dtype
- Is faster than saving to CSV

In [13]:
save_folder="processed_data"

In [14]:
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

np.save(os.path.join(save_folder, "X_lstm.npy"), X_lstm)
np.save(os.path.join(save_folder, "Y_lstm.npy"), Y_lstm)
np.save(os.path.join(save_folder, "X_cls.npy"), X_cls)
np.save(os.path.join(save_folder, "Y_cls.npy"), Y_cls)

In [15]:
df

,timestamp,frame,px,py,pvx,pvy,a_attack,a_dash,a_idle,aim
0,0.020,0,0.500000,0.500000,0.5,0.5000,0,0,1,0.002489
1,0.020,0,0.500000,0.500000,0.5,0.5000,0,0,1,0.005468
2,0.068,1,0.500000,0.496157,0.5,0.1875,0,0,0,0.005197
3,0.068,1,0.500000,0.500000,0.5,0.5000,0,0,1,0.002219
4,0.101,2,0.500000,0.500000,0.5,0.5000,0,0,1,0.002199
...,...,...,...,...,...,...,...,...,...,...
1565,27.200,815,0.309114,0.418996,0.5,0.5000,0,0,1,0.004619
1566,27.233,816,0.309114,0.418996,0.5,0.5000,0,0,1,0.004601
1567,27.266,817,0.309114,0.418996,0.5,0.5000,0,0,1,0.004586
1568,27.301,818,0.309114,0.418996,0.5,0.5000,0,0,1,0.004572
